# 📘 Notebook 08: MapReduce by Hand

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module C: Too Big for One Machine · Notebook 2 of 4 in this module · (8 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain the **MapReduce** model: split, map, shuffle, reduce
- Write a complete MapReduce engine in a page of Python, and run jobs on it
- Express counting, grouping, averaging and joining as a **mapper** and a **reducer**
- Reduce the data sent between machines with a **combiner**, and measure the saving
- Explain how the model survives the **failure** of a machine, and what it demands of your functions in return
- Recognise **data skew**, and say when a distributed job is slower than a simple loop

> **Prerequisites:** Notebook 07 (processing data in pieces and combining small results), Notebook 02 (grouping and joins), Notebook 06 (hashing keys to machines).
>
> 🔭 **Why this notebook matters:** in 2004 two engineers at Google published a short paper describing how the company processed the entire web on thousands of cheap, unreliable computers. Its central observation was that a huge range of computations can be written as just two small functions, and that if a programmer supplies those two, a framework can take care of everything difficult: splitting the data, running on many machines, moving results between them, and recovering when machines break. That paper led to Hadoop, and Hadoop to Spark and to most of what is called "big data". The idea is simple enough to build in an afternoon, which is what we shall do.

> 📦 **Libraries used in this notebook.** None that need installing. The engine is written in plain Python. One cell near the end uses the standard module **`concurrent.futures`** to run workers on several processor cores at once.

In [ ]:
import datetime
import random
import time
import zlib
from collections import defaultdict

## 1. The idea

### Intuition first
An election is held, and ten million paper ballots must be counted by hand. Nobody suggests that one person count them all. What actually happens is this.

1. The ballots are **split** into piles, and each pile goes to a different counting table.
2. At each table, a clerk looks at one ballot at a time and calls out which party it is for. This step needs no knowledge of any other ballot.
3. The ballots are **sorted into trays**, one tray per party. All the ballots for one party end up together, whichever table they came from.
4. For each tray, someone **counts** it and writes down one number.

Steps 2 and 4 are the only ones that have anything to do with elections. Steps 1 and 3 are pure logistics, the same for any counting job. MapReduce is this procedure, with the logistics done by software.

### Formal definition
A **MapReduce** job is defined by two functions.

- The **mapper** takes one input record and produces zero or more **(key, value)** pairs. It sees one record at a time and knows nothing of the others.
- The **reducer** takes one key together with **all** the values produced for that key, and produces a result.

Between them the framework performs the **shuffle**: it collects all pairs with the same key in one place, whichever machine produced them.

```
 input     split        map              shuffle             reduce        output
                                     (group by key)

 records ─► chunk 1 ─► mapper ─► (k, v) pairs ─┐
         ─► chunk 2 ─► mapper ─► (k, v) pairs ─┼─► k1: [v, v, v] ─► reducer ─► result for k1
         ─► chunk 3 ─► mapper ─► (k, v) pairs ─┘   k2: [v, v]    ─► reducer ─► result for k2
```

## 2. Word count, phase by phase

The traditional first example counts how often each word occurs in a collection of text. Here the collection is six short lines, pretended to be spread over three machines.

In [ ]:
lines = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat saw the dog",
    "a dog and a cat",
    "the log and the mat",
    "a cat on a mat",
]

chunks = [lines[0:2], lines[2:4], lines[4:6]]          # split: two lines on each of three machines
for number, chunk in enumerate(chunks):
    print(f"machine {number}: {chunk}")

**Map.** For each line, emit the pair `(word, 1)` for every word in it. The mapper does not count anything. It only labels.

In [ ]:
def word_mapper(line):
    for word in line.split():
        yield (word, 1)

mapped = []
for number, chunk in enumerate(chunks):
    pairs = [pair for line in chunk for pair in word_mapper(line)]
    mapped.append(pairs)
    print(f"machine {number}: {pairs}")

**Shuffle.** Bring together all the values that share a key. This is the step that needs the machines to talk to each other.

In [ ]:
grouped = defaultdict(list)
for pairs in mapped:
    for key, value in pairs:
        grouped[key].append(value)

for key in sorted(grouped):
    print(f"{key:<4} {grouped[key]}")

**Reduce.** For each key, turn its list of values into a result. Here, add them up.

In [ ]:
def sum_reducer(key, values):
    return sum(values)

for key in sorted(grouped):
    print(f"{key:<4} {sum_reducer(key, grouped[key])}")

That is the whole of MapReduce. Notice which parts mention words and counting: the two small functions, six lines between them. Everything else was moving data about.

## 3. A MapReduce engine

Let us package the logistics into one function, so that a job is nothing more than a mapper and a reducer.

Two details make it resemble the real thing. The input is cut into chunks, which stand for the machines that hold the data. And there are several **reducers**, each responsible for a share of the keys. Which reducer gets a key is decided by hashing the key, exactly as in the sharding of Notebook 06, so that every pair with the same key reaches the same reducer.

In [ ]:
def partition_of(key, reducers):
    return zlib.crc32(repr(key).encode()) % reducers          # the same key always goes to the same reducer

def map_reduce(records, mapper, reducer, chunks=4, reducers=3, combiner=None, report=None):
    records = list(records)
    size = max(1, -(-len(records) // chunks))                 # split the input into roughly equal chunks
    pieces = [records[i:i + size] for i in range(0, len(records), size)]

    # --- map: each chunk is processed on its own, as if on a separate machine
    outputs = []
    for piece in pieces:
        pairs = [pair for record in piece for pair in mapper(record)]
        if combiner is not None:                              # optional: summarise locally before sending
            local = defaultdict(list)
            for key, value in pairs:
                local[key].append(value)
            pairs = [(key, combiner(key, values)) for key, values in local.items()]
        outputs.append(pairs)

    # --- shuffle: send every pair to the reducer responsible for its key
    inboxes = [defaultdict(list) for i in range(reducers)]
    shuffled = 0
    for pairs in outputs:
        for key, value in pairs:
            inboxes[partition_of(key, reducers)][key].append(value)
            shuffled += 1

    # --- reduce: each reducer handles its own keys
    results = {}
    for inbox in inboxes:
        for key, values in inbox.items():
            results[key] = reducer(key, values)

    if report is not None:
        report["pairs shuffled"] = shuffled
        report["keys per reducer"] = [len(inbox) for inbox in inboxes]
        report["values per reducer"] = [sum(len(values) for values in inbox.values()) for inbox in inboxes]
    return results

counts = map_reduce(lines, word_mapper, sum_reducer, chunks=3)
print(dict(sorted(counts.items(), key=lambda item: -item[1])))

The same answer as before. From here on, every job in this notebook is two small functions handed to `map_reduce`.

### Real data
The bookshop once more: 200,000 order lines, each a dictionary.

In [ ]:
def make_sales(lines=200_000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    first_day = datetime.date(2023, 1, 1)
    sales = []
    for line_id in range(1, lines + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        quantity = rng.randint(1, 3)
        sales.append({"line_id": line_id, "date": day.isoformat(), "customer": rng.randint(1, 5000),
                      "city": rng.choice(cities), "category": rng.choice(categories),
                      "quantity": quantity, "revenue": round(quantity * rng.uniform(5, 60), 2)})
    return sales

sales = make_sales()
print(f"{len(sales):,} order lines")
print(sales[0])

**Revenue by category.** In SQL: `SELECT category, SUM(revenue) ... GROUP BY category`. In MapReduce, the mapper chooses the **key to group by** and the **value to aggregate**, and the reducer does the aggregation.

In [ ]:
def revenue_mapper(sale):
    yield (sale["category"], sale["revenue"])

revenue = map_reduce(sales, revenue_mapper, sum_reducer)
for category, total in sorted(revenue.items(), key=lambda item: -item[1])[:4]:
    print(f"{category:<10} {total:>14,.2f}")

**A filter and a two-part key.** A mapper may emit nothing for a record, which is how rows are filtered out, and a key may be a tuple.

In [ ]:
def monthly_athens_mapper(sale):
    if sale["city"] == "Athens":                              # WHERE city = 'Athens'
        yield ((sale["date"][:7], sale["category"]), sale["quantity"])        # GROUP BY month, category

copies = map_reduce(sales, monthly_athens_mapper, sum_reducer)
for key in sorted(copies)[:5]:
    print(key, copies[key])
print(f"... {len(copies)} groups")

The correspondence with SQL is exact, and worth fixing in mind.

| SQL | MapReduce |
|---|---|
| `WHERE` | the mapper emits nothing for rows that do not qualify |
| `GROUP BY` columns | the **key** emitted by the mapper |
| the column being aggregated | the **value** emitted by the mapper |
| `SUM`, `COUNT`, `MAX` ... | the reducer |

## 4. The average trap

What is the average revenue of an order line in each city? It is tempting to reuse the pattern: emit `(city, revenue)` and let the reducer take the mean of its values.

In [ ]:
def city_revenue_mapper(sale):
    yield (sale["city"], sale["revenue"])

def mean_reducer(key, values):
    return sum(values) / len(values)

averages = map_reduce(sales, city_revenue_mapper, mean_reducer)
print({city: round(value, 2) for city, value in sorted(averages.items())[:3]})

This is correct, and it hides a trap that is sprung by the optimisation of the next section. Suppose each machine first averages its own share of the data, to send less. Three machines hold the following values for one city.

In [ ]:
machine_values = [[10.0, 20.0, 30.0, 40.0], [50.0], [60.0, 70.0]]

true_average = sum(sum(values) for values in machine_values) / sum(len(values) for values in machine_values)
average_of_averages = sum(sum(values) / len(values) for values in machine_values) / len(machine_values)

print("True average:        ", true_average)
print("Average of averages: ", round(average_of_averages, 2))

The two differ. The machine holding a single value has been given the same weight as the machine holding four. **An average of averages is not the average**, unless every group has the same size.

The cure is the idea from the exercises of Notebook 07. An average cannot be combined, but the two numbers it is made of can. Carry the **sum** and the **count** separately, add them freely, and divide only at the very end.

In [ ]:
def sum_and_count_mapper(sale):
    yield (sale["city"], (sale["revenue"], 1))                # the value is a pair: (sum so far, count so far)

def add_pairs(key, values):
    return (sum(total for total, count in values), sum(count for total, count in values))

def average_reducer(key, values):
    total, count = add_pairs(key, values)
    return total / count

safe_averages = map_reduce(sales, sum_and_count_mapper, average_reducer)
print({city: round(value, 2) for city, value in sorted(safe_averages.items())[:3]})

## 5. Sending less: the combiner

### Intuition first
In a real cluster the expensive part of a job is the **shuffle**. Mapping happens where the data already is. The shuffle sends data across the network, and the network is far slower than a processor.

Look at what our word count sends. The line *"the cat sat on the mat"* produces `(the, 1)` twice, and a large file would send `(the, 1)` millions of times, so that a reducer far away can add up a million ones. It would be much better for each machine to do its own adding first and send `(the, 1000000)` once.

A function that summarises the pairs of **one machine** before they are shuffled is called a **combiner**.

In [ ]:
without, with_combiner = {}, {}

map_reduce(sales, revenue_mapper, sum_reducer, chunks=8, report=without)
result = map_reduce(sales, revenue_mapper, sum_reducer, chunks=8, combiner=sum_reducer, report=with_combiner)

print(f"Pairs shuffled without a combiner: {without['pairs shuffled']:>8,}")
print(f"Pairs shuffled with a combiner:    {with_combiner['pairs shuffled']:>8,}")
print("Same result:", {k: round(v, 2) for k, v in result.items()} == {k: round(v, 2) for k, v in revenue.items()})

Two hundred thousand pairs became sixty-four: eight categories from each of eight machines. The result is identical.

### When is a combiner allowed?
For a sum, the combiner is the reducer itself, because a sum of partial sums is the sum. That is not true of every function, as Section 4 has just shown: a mean of partial means is wrong.

A combiner is safe when the operation can be applied to pieces in any grouping and any order with the same final result. Mathematicians call such an operation **associative** and **commutative**. Sums, counts, minimums and maximums qualify. Averages and medians do not. The sum-and-count trick is the standard way to turn an average into something that does.

In [ ]:
report = {}
averages_combined = map_reduce(sales, sum_and_count_mapper, average_reducer, chunks=8, combiner=add_pairs, report=report)

print("Pairs shuffled:", report["pairs shuffled"])
print("Same as before:", all(abs(averages_combined[city] - safe_averages[city]) < 1e-9 for city in safe_averages))

> ⚠️ **Common pitfalls**
>
> - Using an average, a median or a count of distinct values as a combiner. The job runs without error and gives a wrong answer.
> - Writing a mapper that depends on anything other than its one record: a global counter, the previous record, the time of day. Mappers run on different machines in no fixed order, and may be run twice (Section 6).
> - Emitting a very large number of distinct keys. If every record has its own key, nothing can be combined and everything is shuffled.

## 6. Machines fail

### Intuition first
With one computer, a hardware failure is a rare event. With five thousand computers, something fails every few hours: it is not an exception, it is the weather. A system that had to start the whole job again whenever one machine died would never finish a long job at all.

MapReduce survives failure by a very plain method. If the machine processing a chunk dies, the framework gives **that chunk** to another machine and runs it again. Nothing else is redone.

Let us add failure to the map phase. Each chunk has some chance of being lost while it is being processed.

In [ ]:
def map_with_failures(records, mapper, chunks=10, failure_rate=0.3, seed=0):
    rng = random.Random(seed)
    size = -(-len(records) // chunks)
    pieces = [records[i:i + size] for i in range(0, len(records), size)]

    outputs, attempts = [], 0
    for number, piece in enumerate(pieces):
        while True:
            attempts += 1
            if rng.random() < failure_rate:                   # the machine died: its work is lost
                print(f"  chunk {number}: machine failed, running the chunk again on another machine")
                continue
            outputs.append([pair for record in piece for pair in mapper(record)])
            break
    return outputs, attempts

outputs, attempts = map_with_failures(sales, revenue_mapper)

totals = defaultdict(float)
for pairs in outputs:
    for key, value in pairs:
        totals[key] += value

print(f"\n{attempts} attempts were needed for 10 chunks")
print("Result still correct:", {k: round(v, 2) for k, v in totals.items()} == {k: round(v, 2) for k, v in revenue.items()})

Wherever a machine failed, its chunk was run again, and the answer is exactly right. Change the `seed` or raise the `failure_rate` and run the cell again: more machines fail, more attempts are needed, and the result does not change.

This works only because of a property we have relied on without stating it. Running a mapper on the same chunk a second time gives **exactly the same output** as the first, and has no other effect on the world. A function like that is called **pure**, or *deterministic and free of side effects*. If our mapper had also added to a running total somewhere, or sent an e-mail, a repeated chunk would have counted twice or sent two e-mails.

This is the bargain at the heart of MapReduce. You agree to write only pure functions of one record. In return, the framework can run them anywhere, in any order, as many times as necessary, and guarantee the result.

## 7. Uneven work: skew

Our reducers were given keys by hashing, which spreads **keys** evenly. It does not spread **work** evenly, if some keys have far more values than others.

In our invented data every city has about the same number of sales. Real data never looks like that. Let us make it realistic: half of all sales come from the capital.

In [ ]:
def skewed(sale, rng):
    if rng.random() < 0.5:
        return dict(sale, city="Athens")
    return sale

rng = random.Random(1)
skewed_sales = [skewed(sale, rng) for sale in sales]

for name, data in [("even data", sales), ("skewed data", skewed_sales)]:
    report = {}
    map_reduce(data, city_revenue_mapper, sum_reducer, reducers=4, report=report)
    values = report["values per reducer"]
    print(f"{name:<12} values handled by each of 4 reducers: {values}   busiest / average = {max(values) / (sum(values) / 4):.1f}")

With skewed data, one reducer receives well over half of everything, because all the values of one key must go to the same place. A job is finished only when its **slowest** part is finished. That reducer decides how long the whole job takes, and the other three machines sit idle.

Adding machines does not help: the Athens values still all go to one of them. This is **data skew**, and in practice it is the commonest reason for a distributed job being mysteriously slow. The remedies all amount to giving the heavy key special treatment: combining before the shuffle, as in Section 5, which here reduces each machine's Athens values to a single pair, or splitting the heavy key artificially into several.

## 8. Truly in parallel

*This section starts several worker processes. It is meant for Colab, where a notebook has two processor cores.*

Everything so far ran the "machines" one after another, to show the logic. Here the map phase really runs on several cores at once. The standard module `concurrent.futures` starts a pool of worker processes and hands each a chunk.

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor

def heavy_mapper(sale):
    total = 0
    for i in range(300):                                      # some real work for each record
        total += (sale["line_id"] * i) % 7
    yield (sale["category"], total)

def map_chunk(chunk):
    local = defaultdict(int)                                  # map and combine, on one worker
    for sale in chunk:
        for key, value in heavy_mapper(sale):
            local[key] += value
    return dict(local)

def run(workers, chunks=8):
    size = -(-len(sales) // chunks)
    pieces = [sales[i:i + size] for i in range(0, len(sales), size)]
    started = time.perf_counter()
    if workers == 1:
        partial_results = [map_chunk(piece) for piece in pieces]
    else:
        with ProcessPoolExecutor(max_workers=workers) as pool:
            partial_results = list(pool.map(map_chunk, pieces))
    totals = defaultdict(int)
    for partial in partial_results:                           # reduce: combine the small results
        for key, value in partial.items():
            totals[key] += value
    return dict(totals), time.perf_counter() - started

print("Processor cores available:", os.cpu_count())
one_result, one_time = run(workers=1)
many_result, many_time = run(workers=os.cpu_count())
print(f"1 worker:  {one_time:.2f} s")
print(f"{os.cpu_count()} workers: {many_time:.2f} s   ({one_time / many_time:.1f} times faster)")
print("Same result:", one_result == many_result)

With two cores the job should run somewhat less than twice as fast. It is never quite twice, because starting the workers, sending them their chunks and collecting the results all take time. Notebook 12 returns to the question of how much speed more workers can buy.

## 9. When is it worth it?

Our engine does a good deal of work that a plain loop does not: it splits the data, builds lists of pairs, hashes every key, and fills inboxes. Let us compare it with the most direct way of computing the same thing.

In [ ]:
def direct(sales):
    totals = defaultdict(float)
    for sale in sales:
        totals[sale["category"]] += sale["revenue"]
    return totals

started = time.perf_counter()
direct(sales)
direct_time = time.perf_counter() - started

started = time.perf_counter()
map_reduce(sales, revenue_mapper, sum_reducer)
engine_time = time.perf_counter() - started

print(f"A plain loop:        {direct_time * 1000:7.1f} ms")
print(f"Our MapReduce engine: {engine_time * 1000:7.1f} ms   ({engine_time / direct_time:.0f} times slower)")

On data that fits on one machine, the plain loop wins easily. All the machinery of MapReduce is **overhead**: work that produces nothing by itself and exists only to make distribution possible. A real cluster adds more of it: starting processes on many machines, copying data over the network, writing intermediate results to disk so that they survive a failure.

| | Data fits on one machine | Data needs many machines |
|---|---|---|
| A plain loop, or a database | fast, simple, easy to debug | cannot run at all |
| MapReduce | the same answer, many times slower | works, and keeps working when machines fail |
| Better choice | **the plain loop** | **MapReduce** |

A much-quoted study of 2015 compared published results from large cluster systems with a single-threaded program on one laptop, on the same tasks. The laptop won most of them. A cluster does not make a computation fast. It makes a computation **possible** when it is too large for one machine, and it pays for that with a great deal of overhead. The first question about any dataset is whether it truly needs one.

## 10. Where we stand

| The problem | The answer |
|---|---|
| More data than one machine can process in the time available | split it, and process the pieces on many machines |
| Each kind of analysis would need its own distributed program | write only a mapper and a reducer, and let a framework do the rest |
| The network is the slowest part | combine locally before shuffling |
| Machines fail constantly | pure functions, so that any piece can simply be run again |
| One key holds most of the data | skew: the busiest worker sets the pace |

MapReduce as you have built it is exactly what Hadoop offered the world in 2006. It was a breakthrough, and it was laborious to use. Every analysis, however small, had to be bent into the shape of one mapper and one reducer. A join, or a query with three steps, meant several jobs in a row, each writing its full results to disk for the next to read back.

The next notebook introduces the system that kept the ideas of this one (split, shuffle, pure functions, re-running after failure) and removed the labour.

---
## ✏️ Exercises

### Exercise 1 (Count)
Using `map_reduce`, count the number of order lines in each city. Then modify the mapper so that only lines with a quantity of 3 are counted.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def city_count_mapper(sale):
    yield (sale["city"], 1)

def triple_count_mapper(sale):
    if sale["quantity"] == 3:
        yield (sale["city"], 1)

print(dict(sorted(map_reduce(sales, city_count_mapper, sum_reducer).items())))
print(dict(sorted(map_reduce(sales, triple_count_mapper, sum_reducer).items())))
```

*Counting is summing ones. A filter is a mapper that sometimes emits nothing. The reducer did not change.*
</details>

### Exercise 2 (The largest sale of each customer)
Find, for each customer, the revenue of their single largest order line. Write the mapper and the reducer. Can the reducer also be used as a combiner? Check, by running the job with and without it and comparing the results and the number of pairs shuffled.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def customer_mapper(sale):
    yield (sale["customer"], sale["revenue"])

def max_reducer(key, values):
    return max(values)

plain, combined = {}, {}
largest = map_reduce(sales, customer_mapper, max_reducer, report=plain)
largest_combined = map_reduce(sales, customer_mapper, max_reducer, combiner=max_reducer, report=combined)

print("Same result:", largest == largest_combined)
print(f"Pairs shuffled: {plain['pairs shuffled']:,} without a combiner, {combined['pairs shuffled']:,} with one")
```

*A maximum of partial maximums is the maximum, so the reducer is a valid combiner. It saves far less here than it did for categories, because there are 5,000 customers: each machine still has to send one pair for nearly every customer. A combiner helps most when there are few keys and many values for each.*
</details>

### Exercise 3 (An index of words)
Search engines keep an **inverted index**: for each word, the list of documents in which it occurs. Build one with MapReduce for the six `lines` of Section 2, using the position of each line as its document number. The input records should be pairs `(number, line)`. For each word, the result should be a sorted list of document numbers without repetitions.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def index_mapper(record):
    number, line = record
    for word in set(line.split()):                 # set: mention each word once per document
        yield (word, number)

def index_reducer(key, values):
    return sorted(set(values))

index = map_reduce(list(enumerate(lines)), index_mapper, index_reducer, chunks=3)
for word in ["cat", "dog", "the", "saw"]:
    print(f"{word:<4} {index[word]}")
```

*The reducer does not have to produce a number. Building the index of the web was the job for which MapReduce was invented.*
</details>

### Exercise 4 (Distinct customers)
How many **different** customers bought something in each category? Write the job. Then explain why "the number of distinct values" cannot be used as a combiner, and what a correct combiner would have to send.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def category_customer_mapper(sale):
    yield (sale["category"], sale["customer"])

def distinct_reducer(key, values):
    return len(set(values))

print(dict(sorted(map_reduce(sales, category_customer_mapper, distinct_reducer).items())))

# A correct combiner: send the SET of customers seen on this machine, not their number.
def set_mapper(sale):
    yield (sale["category"], {sale["customer"]})

def union(key, values):
    return set().union(*values)

def size_of_union(key, values):
    return len(union(key, values))

report = {}
combined = map_reduce(sales, set_mapper, size_of_union, combiner=union, report=report)
print(dict(sorted(combined.items())), "  pairs shuffled:", report["pairs shuffled"])
```

*If two machines each report "1,000 distinct customers", the total could be anything from 1,000 to 2,000, depending on how many they have in common. The counts cannot be added. The sets can be united, so a correct combiner sends the set itself. Fewer pairs are shuffled, but each pair now carries thousands of customers: counting distinct values is expensive in every distributed system, and is one of the commonest uses of approximate methods.*
</details>

### Exercise 5 (A join, a little harder)
Joins can be done with MapReduce too. Below is a small table of customers. Join it with the sales to compute the total revenue for each **age group**. The trick: put both tables into one input, with each record tagged by where it came from, and use the customer id as the key. The reducer for a customer then receives that customer's age group together with all their sales. This needs two jobs: one for the join, and a second to add up by age group.

In [ ]:
rng = random.Random(5)
customer_table = [{"customer": i, "age_group": rng.choice(["under 30", "30 to 49", "50 and over"])} for i in range(1, 5001)]
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
tagged = [("customer", row) for row in customer_table] + [("sale", row) for row in sales]

def join_mapper(record):
    source, row = record
    if source == "customer":
        yield (row["customer"], ("age_group", row["age_group"]))
    else:
        yield (row["customer"], ("revenue", row["revenue"]))

def join_reducer(key, values):
    age_group = [value for tag, value in values if tag == "age_group"][0]
    revenue = sum(value for tag, value in values if tag == "revenue")
    return (age_group, revenue)

joined = map_reduce(tagged, join_mapper, join_reducer)             # job 1: one (age group, revenue) per customer

def regroup_mapper(record):
    age_group, revenue = record
    yield (age_group, revenue)

by_age = map_reduce(list(joined.values()), regroup_mapper, sum_reducer)          # job 2: add up by age group
for age_group, revenue in sorted(by_age.items()):
    print(f"{age_group:<12} {revenue:>14,.2f}")

check = defaultdict(float)                                         # the direct way, to check
age_of = {row["customer"]: row["age_group"] for row in customer_table}
for sale in sales:
    check[age_of[sale["customer"]]] += sale["revenue"]
print("Correct:", all(abs(by_age[group] - check[group]) < 0.01 for group in check))
```

*The shuffle did the join: by using the customer id as the key, it brought each customer's description and their sales to the same reducer. This is called a reduce-side join. Notice how much ceremony one line of SQL required here: tagging, two jobs, and an intermediate result. That labour is exactly what the next notebook removes.*
</details>

## 🔑 Key takeaways

- **MapReduce** expresses a computation as a **mapper** (one record to key-value pairs) and a **reducer** (one key and all its values to a result). The framework splits the input, **shuffles** pairs by key, and runs everything on many machines.
- The mapper's key is the `GROUP BY`. The reducer is the aggregate.
- A **combiner** summarises on each machine before the shuffle. It must be **associative and commutative**: sums and maximums, yes. Averages, no: carry a sum and a count.
- Failures are handled by **running a chunk again**, which is safe only if mappers and reducers are **pure functions**.
- **Skew**: all the values of one key go to one reducer, and the busiest reducer sets the pace.
- Distribution adds **overhead**. On data that fits on one machine, a plain loop or a database is faster.

---
**Next:** *Notebook 09: Spark: RDDs and DataFrames*, where the same ideas return in a far more comfortable form, and a real cluster engine runs inside your notebook.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*